# S&P 500 Market Map Replica

This notebook is a public, self-contained replica of the market-map project workflow.

It does three things in one place:

- builds the current S&P 500 company universe from Wikipedia
- fetches market cap and historical prices from Yahoo Finance via `yfinance`
- reproduces the market-cap and return treemaps in separate notebook cells

Important note:

- this notebook uses the **current** S&P 500 constituents and looks backward in time
- it is not a historical point-in-time reconstruction of index membership


## Cell 1: Environment, paths, and outputs

This cell:

- detects the project root without absolute paths
- defines the output files
- prepares the output directory


In [ ]:
from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests
import yfinance as yf
from bs4 import BeautifulSoup

cwd = Path.cwd().resolve()

def looks_like_project_root(path: Path) -> bool:
    return (path / 'data').exists() and (path / 'notebooks').exists()

candidates = [cwd, *cwd.parents]
ROOT = next((path for path in candidates if looks_like_project_root(path)), cwd)

if ROOT == cwd and not looks_like_project_root(ROOT):
    print(f'Warning: project root not detected. Using current working directory: {ROOT}')

OUT_DIR = ROOT / 'data' / 'manual' / 'notebook'
OUT_DIR.mkdir(parents=True, exist_ok=True)

WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

CONSTITUENTS_OUT = OUT_DIR / 'sp500_current_constituents.csv'
WIDE_OUT = OUT_DIR / 'sp500_timeseries_15y_wide.csv'
LONG_OUT = OUT_DIR / 'sp500_timeseries_15y_long.csv'
META_OUT = OUT_DIR / 'sp500_timeseries_15y_meta.json'
MARKET_MAP_OUT = OUT_DIR / 'sp500_market_map_replica.csv'

MARKET_CAP_HTML = OUT_DIR / 'sp500_market_cap_treemap.html'
RETURN_1D_HTML = OUT_DIR / 'sp500_return_1d_treemap.html'
RETURN_YTD_HTML = OUT_DIR / 'sp500_return_ytd_treemap.html'
RETURN_1Y_HTML = OUT_DIR / 'sp500_return_1y_treemap.html'
RETURN_5Y_HTML = OUT_DIR / 'sp500_return_5y_treemap.html'
RETURN_10Y_HTML = OUT_DIR / 'sp500_return_10y_treemap.html'

print('ROOT =', ROOT)
print('OUT_DIR =', OUT_DIR)


## Cell 2: Download the current S&P 500 constituents from Wikipedia

This cell extracts:

- `ticker`
- `name`
- `sector`
- `industry`

It also writes a CSV snapshot of the current constituents.


In [ ]:
def normalize_ticker(symbol: str) -> str:
    return symbol.strip().replace('.', '-')


resp = requests.get(
    WIKI_URL,
    headers={'User-Agent': 'Mozilla/5.0'},
    timeout=30,
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')
table = soup.select_one('#constituents')
if table is None:
    raise ValueError('Could not find Wikipedia constituents table')

rows = []
for tr in table.select('tbody tr'):
    tds = [td.get_text(' ', strip=True) for td in tr.select('td')]
    if len(tds) < 4:
        continue

    rows.append({
        'ticker': normalize_ticker(tds[0]),
        'name': tds[1],
        'sector': tds[2],
        'industry': tds[3],
    })

constituents_df = pd.DataFrame(rows)
constituents_df.to_csv(CONSTITUENTS_OUT, index=False)
constituents_df.head()


## Cell 3: Download market cap and calculate current participation

This cell builds the current composition layer of the market map.

It:

- queries Yahoo Finance for `marketCap`
- merges that data with the Wikipedia classification
- computes `weight = marketCap / totalMarketCap * 100`


In [ ]:
tickers = (
    constituents_df['ticker']
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

market_caps = []
for ticker in tickers:
    market_cap = None

    try:
        instrument = yf.Ticker(ticker)

        try:
            market_cap = instrument.fast_info.get('market_cap')
        except Exception:
            market_cap = None

        if market_cap is None:
            try:
                info = instrument.info
                market_cap = info.get('marketCap')
            except Exception:
                market_cap = None
    except Exception:
        market_cap = None

    market_caps.append({
        'ticker': ticker,
        'marketCap': market_cap,
    })

    time.sleep(0.2)

caps_df = pd.DataFrame(market_caps)
market_cap_df = constituents_df.merge(caps_df, on='ticker', how='left')
market_cap_df['marketCap'] = pd.to_numeric(market_cap_df['marketCap'], errors='coerce')
market_cap_df = market_cap_df[market_cap_df['marketCap'].notna() & (market_cap_df['marketCap'] > 0)].copy()

total_market_cap = market_cap_df['marketCap'].sum()
market_cap_df['weight'] = market_cap_df['marketCap'] / total_market_cap * 100
market_cap_df = market_cap_df.sort_values('weight', ascending=False).reset_index(drop=True)

market_cap_df[['ticker', 'name', 'sector', 'industry', 'marketCap', 'weight']].head(10)


## Cell 4: Download 15 years of adjusted daily prices

This cell requests up to 15 years of adjusted close history for the current constituents.
That history is later used to calculate the return horizons shown in the web app.


In [ ]:
history = yf.download(
    tickers=tickers,
    period='15y',
    interval='1d',
    auto_adjust=True,
    group_by='ticker',
    threads=True,
    progress=False,
)

history.shape


## Cell 5: Build reusable `wide`, `long`, and `meta` outputs

This cell turns the raw Yahoo download into:

- `wide`: one row per date, one column per ticker
- `long`: one row per date-ticker observation
- `meta`: a JSON summary of the extraction


In [ ]:
def extract_close_wide(frame: pd.DataFrame, tickers: list[str]) -> tuple[pd.DataFrame, list[str]]:
    if frame.empty:
        return pd.DataFrame(), tickers

    close_map: dict[str, pd.Series] = {}
    missing: list[str] = []

    if isinstance(frame.columns, pd.MultiIndex):
        available = set(frame.columns.get_level_values(0))
        for ticker in tickers:
            if ticker not in available:
                missing.append(ticker)
                continue

            sub = frame[ticker]
            if 'Close' not in sub:
                missing.append(ticker)
                continue

            series = sub['Close'].copy()
            if series.dropna().empty:
                missing.append(ticker)
                continue

            close_map[ticker] = series
    else:
        if 'Close' in frame:
            close_map[tickers[0]] = frame['Close'].copy()
        else:
            missing.extend(tickers)

    wide = pd.DataFrame(close_map)
    wide.index = pd.to_datetime(wide.index)
    wide.index.name = 'date'
    wide = wide.sort_index()
    return wide, missing


close_wide, missing_tickers = extract_close_wide(history, tickers)

close_long = (
    close_wide.reset_index()
    .melt(id_vars='date', var_name='ticker', value_name='adj_close')
    .dropna(subset=['adj_close'])
    .merge(constituents_df[['ticker', 'name', 'sector', 'industry']], on='ticker', how='left')
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

meta = {
    'generatedAt': datetime.now(timezone.utc).isoformat(),
    'source': 'Current S&P 500 constituents from Wikipedia',
    'sourceUrl': WIKI_URL,
    'period': '15y',
    'interval': '1d',
    'autoAdjust': True,
    'tickersRequested': len(tickers),
    'tickersReturned': int(close_wide.shape[1]),
    'missingTickers': missing_tickers,
    'wideShape': [int(close_wide.shape[0]), int(close_wide.shape[1])],
    'longShape': [int(close_long.shape[0]), int(close_long.shape[1])],
    'note': 'Current constituents with backward-looking adjusted price history. This is not historical point-in-time index membership.',
}

close_wide.to_csv(WIDE_OUT)
close_long.to_csv(LONG_OUT, index=False)
META_OUT.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

close_wide.shape, close_long.shape, len(missing_tickers)


## Cell 6: Calculate `1D`, `YTD`, `1Y`, `5Y`, and `10Y` returns

This cell combines the composition layer with the price-history layer.

It produces the final analysis table used by the treemaps:

- `weight`
- `return_1d`
- `return_ytd`
- `return_1y`
- `return_5y`
- `return_10y`


In [ ]:
def pct_return(current: float | None, base: float | None) -> float | None:
    if current is None or base is None or base == 0:
        return None
    return (current / base - 1) * 100


def asof_value(series: pd.Series, target: pd.Timestamp | None = None) -> float | None:
    clean = series.dropna().sort_index()
    if clean.empty:
        return None

    if target is None:
        return float(clean.iloc[-1])

    subset = clean.loc[:target]
    if subset.empty:
        return None
    return float(subset.iloc[-1])


def first_value_in_year(series: pd.Series, year: int) -> float | None:
    clean = series.dropna().sort_index()
    if clean.empty:
        return None

    subset = clean[clean.index.year == year]
    if subset.empty:
        return None
    return float(subset.iloc[0])


return_rows = []
for ticker in close_wide.columns:
    series = close_wide[ticker].dropna().sort_index()
    if series.empty:
        continue

    latest_value = float(series.iloc[-1])
    latest_date = pd.Timestamp(series.index[-1])

    prev_value = float(series.iloc[-2]) if len(series) >= 2 else None
    ytd_base = first_value_in_year(series, latest_date.year)
    base_1y = asof_value(series, latest_date - pd.DateOffset(years=1))
    base_5y = asof_value(series, latest_date - pd.DateOffset(years=5))
    base_10y = asof_value(series, latest_date - pd.DateOffset(years=10))

    return_rows.append({
        'ticker': ticker,
        'return_1d': pct_return(latest_value, prev_value),
        'return_ytd': pct_return(latest_value, ytd_base),
        'return_1y': pct_return(latest_value, base_1y),
        'return_5y': pct_return(latest_value, base_5y),
        'return_10y': pct_return(latest_value, base_10y),
    })

returns_df = pd.DataFrame(return_rows)
market_map_df = market_cap_df.merge(returns_df, on='ticker', how='left')
market_map_df = market_map_df.sort_values('weight', ascending=False).reset_index(drop=True)
market_map_df.to_csv(MARKET_MAP_OUT, index=False)

market_map_df.head(10)


## Cell 7: Plot helpers

These helpers are used by the plot cells below.

Each figure is shown inline in its own cell.
Each cell can also export the corresponding HTML artifact.


In [ ]:
SECTOR_COLORS = {
    'Information Technology': '#78b9ff',
    'Financials': '#22c55e',
    'Communication Services': '#b793ff',
    'Consumer Discretionary': '#8395ff',
    'Health Care': '#e7e3ff',
    'Industrials': '#ff9cbc',
    'Consumer Staples': '#ff985f',
    'Energy': '#f4c84d',
    'Utilities': '#d8df53',
    'Materials': '#ff6767',
    'Real Estate': '#b7ece0',
}

RETURN_SCALES = [
    [0.0, '#ff3b30'],
    [0.5, '#1b1b1b'],
    [1.0, '#39ff14'],
]


def build_market_cap_treemap(df: pd.DataFrame):
    fig = px.treemap(
        df,
        path=['sector', 'industry', 'ticker'],
        values='weight',
        color='sector',
        color_discrete_map=SECTOR_COLORS,
        custom_data=['name', 'weight', 'sector', 'industry', 'marketCap'],
    )

    fig.update_traces(
        texttemplate='%{label}<br>%{value:.2f}%',
        hovertemplate=(
            '<b>%{label}</b><br>'
            'Company: %{customdata[0]}<br>'
            'Weight: %{customdata[1]:.2f}%<br>'
            'Sector: %{customdata[2]}<br>'
            'Industry: %{customdata[3]}<br>'
            'Market Cap: %{customdata[4]:,.0f}<extra></extra>'
        ),
        marker_line_width=1,
    )

    fig.update_layout(
        title='S&P 500 Market Cap Treemap',
        paper_bgcolor='#0a0908',
        plot_bgcolor='#0a0908',
        font=dict(color='white', size=14),
        margin=dict(t=70, l=10, r=10, b=10),
    )

    return fig


def build_return_treemap(df: pd.DataFrame, column: str, title: str):
    chart_df = df.copy()
    abs_max = chart_df[column].dropna().abs().quantile(0.95)
    if pd.isna(abs_max) or abs_max == 0:
        abs_max = 1.0

    chart_df['color_metric'] = chart_df[column].clip(-abs_max, abs_max)

    fig = px.treemap(
        chart_df,
        path=['sector', 'industry', 'ticker'],
        values='weight',
        color='color_metric',
        color_continuous_scale=RETURN_SCALES,
        color_continuous_midpoint=0,
        custom_data=['name', 'weight', 'sector', 'industry', column],
    )

    fig.update_traces(
        texttemplate='%{label}',
        hovertemplate=(
            '<b>%{label}</b><br>'
            'Company: %{customdata[0]}<br>'
            'Weight: %{customdata[1]:.2f}%<br>'
            'Sector: %{customdata[2]}<br>'
            'Industry: %{customdata[3]}<br>'
            'Return: %{customdata[4]:.2f}%<extra></extra>'
        ),
        marker_line_width=1,
    )

    fig.update_layout(
        title=title,
        paper_bgcolor='#0a0908',
        plot_bgcolor='#0a0908',
        font=dict(color='white', size=14),
        margin=dict(t=70, l=10, r=10, b=10),
        coloraxis_colorbar=dict(title='Return %'),
    )

    return fig


## Cell 8: Market-cap treemap

This is the composition version of the chart.
Box size is `weight`, and color is the sector palette.


In [ ]:
market_cap_fig = build_market_cap_treemap(market_map_df)
market_cap_fig.write_html(MARKET_CAP_HTML, include_plotlyjs='cdn')
market_cap_fig


## Cell 9: 1D return treemap

This chart uses the same box sizes as the composition map, but colors each company by its 1-day return.


In [ ]:
return_1d_fig = build_return_treemap(market_map_df, 'return_1d', 'S&P 500 Treemap · 1D Return')
return_1d_fig.write_html(RETURN_1D_HTML, include_plotlyjs='cdn')
return_1d_fig


## Cell 10: YTD return treemap

This chart colors each company by its year-to-date return.


In [ ]:
return_ytd_fig = build_return_treemap(market_map_df, 'return_ytd', 'S&P 500 Treemap · YTD Return')
return_ytd_fig.write_html(RETURN_YTD_HTML, include_plotlyjs='cdn')
return_ytd_fig


## Cell 11: 1Y return treemap

This chart colors each company by its trailing 1-year return.


In [ ]:
return_1y_fig = build_return_treemap(market_map_df, 'return_1y', 'S&P 500 Treemap · 1Y Return')
return_1y_fig.write_html(RETURN_1Y_HTML, include_plotlyjs='cdn')
return_1y_fig


## Cell 12: 5Y return treemap

This chart colors each company by its trailing 5-year return.


In [ ]:
return_5y_fig = build_return_treemap(market_map_df, 'return_5y', 'S&P 500 Treemap · 5Y Return')
return_5y_fig.write_html(RETURN_5Y_HTML, include_plotlyjs='cdn')
return_5y_fig


## Cell 13: 10Y return treemap

This chart colors each company by its trailing 10-year return.


In [ ]:
return_10y_fig = build_return_treemap(market_map_df, 'return_10y', 'S&P 500 Treemap · 10Y Return')
return_10y_fig.write_html(RETURN_10Y_HTML, include_plotlyjs='cdn')
return_10y_fig


## Cell 14: Saved outputs summary

This final cell only reports what was written to disk.


In [ ]:
print('Saved:')
print('-', CONSTITUENTS_OUT)
print('-', WIDE_OUT)
print('-', LONG_OUT)
print('-', META_OUT)
print('-', MARKET_MAP_OUT)
print('-', MARKET_CAP_HTML)
print('-', RETURN_1D_HTML)
print('-', RETURN_YTD_HTML)
print('-', RETURN_1Y_HTML)
print('-', RETURN_5Y_HTML)
print('-', RETURN_10Y_HTML)
